# Workshop Quiz on Robotics Lesson 10 - "Autonomous Driving with Computer Vision"

This is the final and most advanced module of this workshop. In Module 10, we will combine techniques from previous lessons to create a functional autonomous driving pipeline. This includes using **computer vision for obstacle detection**, **estimating depth from a single camera**, and **integrating that data into a SLAM (Simultaneous Localization and Mapping) framework** to allow our rover to localize itself and navigate in real-time.


By the end of this module, your robot will be able to:

1. **Interpret** visual input to classify objects and hazards in the environment

2. **Estimate** distances using a single camera (monocular depth estimation)

3. **Build** and **update** a map using computer vision (vision-based SLAM)

4. **Make** driving decisions using A* pathfinding with real-time heuristics

We're going to be bringing it all together from the prior modules:

- **Module 7**: Sensor Fusion and Feedback Control

- **Module 8**: Mapping, Odometry, and SLAM

- **Module 9**: Computer Vision and AI Object Detection

# Before continuing, run the cell below.

In [2]:
import module10
from module10 import functions_dict
def check_answer(question_id, charlie):
    if charlie.strip() == "":
        print("Please write your answer inside the quotations above.")
        return
    alpha = functions_dict.get(question_id, None)
    if alpha is None:
        print(f"Question ID '{question_id}' not found.")
    else:
        beta = alpha["answer"]
        if charlie.upper() == beta:
            print("Correct!")
        else:
            print("Incorrect. Hint:", alpha["hint"])

## Depth Estimation with Vision

### What is Depth Estimation?

**Depth estimation** is the process of **determining the distance** from a camera to objects in the scene. It’s what allows a robot to “**see**” in 3D. Overall, it supports **navigation and obstacle avoidance** and enhances the mapping process.

### Why Does Depth Estimation Matter?

- **Navigation & Obstacle avoidance**: Robots need to understand the distance to obstacle so that they can plan safe paths 

- **Manipulation**: Depth data helps robotic arms accurately grasp and interact with objects 

- **Mapping and Localization**: Depth information contributes to creating 3D maps and aids in simultaneous localization and mapping (SLAM). This is essential for autonomous robot.

### Different Methods of Vision-Based Depth Estimation:

We will primarily focus on the first of these in our demonstration construction but wanted to provide additional pathways for you to explore.

1. **Monocular Depth Estimation (Single Camera + Machine Learning)**

 **How it works:**
 
This method uses a **single 2D image** from a regular camera and applies a deep learning model (**MiDaS-v2**) to estimate depth. The model has been trained on thousands of scenes to learn patterns that suggest distance — like perspective, object size, and shading.

    - Example: MiDaS-v2

    - Pros: Low hardware cost, easy to deploy

    - Cons: Lower precision vs stereo methods
    
Perfect for robots that just need to know “what’s in front of me?” and “how far apart are things?” in **relative terms**.


2. **Stereo Vision (Two Cameras)**

**How it works:**

This method **mimics how human vision works**. It uses two cameras placed a short distance apart, and compares the disparity (difference) between the left and right images to calculate depth. The greater the disparity between object positions, the closer they are.

    - Calculates depth based on disparity between camera images

    - Pros: More accurate

    - Cons: Needs extra hardware and calibration
    
Useful for detailed mapping and tasks where **precision matters**, like robotic arms or SLAM in larger spaces.

3. **Active Sensors (LiDAR, ToF)**

**How it works:**

Active sensors emit light (like laser or infrared) and measure how long it takes to reflect back. This direct measurement gives accurate depth readings. LiDAR scans across many points, while ToF sensors measure a single distance or depth image.

    - Not used in this module, but helpful reference

    - Pros: Very accurate

    - Cons: Expensive, more power-hungry

Used in **professional robotics, autonomous vehicles, and industrial SLAM systems** — not necessary for our workshop, but important to understand.


Each of these methods has its **unique advantages and limitations**. The choice among them depends on factors like required accuracy, budget constraints, environmental conditions, and specific application needs.

### Question 1: Which depth estimation method uses a single camera and machine learning?
​ 
**A.** LiDAR  
**B.** Stereo Vision  
**C.** Monocular Depth Estimation  
**D.** Infrared Mapping   

In [ ]:
answer_1 = ""

check_answer("Q1", answer_1)

### Question 2: True or False: Stereo vision requires calibration of two cameras placed at different positions.

​ 
**A.** True  
**B.** False  

In [ ]:
answer_2 = ""

check_answer("Q2", answer_2)

## Heuristic Assignment for Object Detection

### What is Heuristic Assignment?

When a robot detects an object (like a chair or a person), it needs to assign a **heuristic value** — a label or cost — to that object to inform pathfinding decisions.

In short: **detection becomes actionable**.

Pathfinding algorithms like **A-Star** don’t understand what a “chair” is — they only know about nodes and cost. So we use heuristic assignment to bridge this gap:

### Heuristic Assignment Table

| Object Detected | Heuristic Type | Example Use in A*         |
|------------------|----------------|----------------------------|
| Wall             | Impassable       | Impassable                 |
| Chair            | High cost      | Avoid if possible          |
| Person           | Impassable       | Must not enter             |
| Floor            | Free space     | Path can pass              |

We classify each object as one of:

- **Impassable** (infinite cost)

- **High-cost area** (like furniture, tight space)

- **Free path** (safe to traverse)


# Code for A* 

```
class Node:
    def __init__(self, x, y, g=0, h=0, parent=None):
        self.x = x
        self.y = y
        self.g = g
        self.h = h
        self.f = g + h
        self.parent = parent

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

def heuristic(a, b):
    return abs(a.x - b.x) + abs(a.y - b.y)

def move_cost(neighbor, terrain_map):
    terrain = terrain_map[neighbor.x][neighbor.y]
    if terrain == "obstacle":
        return float('inf')
    elif terrain == "difficult":
        return 999
    else:
        return 1

def get_neighbors(node, terrain_map):
    neighbors = []
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    rows = len(terrain_map)
    cols = len(terrain_map[0])
    for dx, dy in moves:
        x2, y2 = node.x + dx, node.y + dy
        if 0 <= x2 < rows and 0 <= y2 < cols:
            if terrain_map[x2][y2] != "obstacle":
                neighbors.append(Node(x2, y2))
    return neighbors

def a_star(start, goal, terrain_map):
    open_list = [start]
    closed_set = set()

    while open_list:
        current = min(open_list, key=lambda node: node.f)
        
        if current == goal:
            path = []
            while current:
                path.append((current.x, current.y))
                current = current.parent
            return path[::-1]
        
        open_list.remove(current)
        closed_set.add((current.x, current.y))
        
        for neighbor in get_neighbors(current, terrain_map):
            if (neighbor.x, neighbor.y) in closed_set:
                continue
            
            tentative_g = current.g + move_cost(neighbor, terrain_map)
            
            in_open = False
            for open_node in open_list:
                if neighbor == open_node:
                    in_open = True
                    if tentative_g < open_node.g:
                        open_node.g = tentative_g
                        open_node.f = open_node.g + open_node.h
                        open_node.parent = current
                    break
            
            if not in_open:
                neighbor.g = tentative_g
                neighbor.h = heuristic(neighbor, goal)
                neighbor.f = neighbor.g + neighbor.h
                neighbor.parent = current
                open_list.append(neighbor)
    
    return None

if __name__ == "__main__":
    terrain_map = [
        ["normal",   "normal",    "normal",    "normal"],
        ["normal",   "obstacle",  "difficult", "normal"],
        ["normal",   "difficult", "normal",    "normal"],
        ["normal",   "normal",    "normal",    "normal"]
    ]

    start = Node(0, 0)
    goal = Node(3, 3)

    path = a_star(start, goal, terrain_map)
```



## How do we do this?

Since we're using a YOLO-based model for object detection, each detection has a label and bounding box. You can assign heuristic values based on that label.

```
def assign_heuristic(label):
    if label in ["wall", "person"]:
        return float("inf")  # impassable
    elif label in ["obstacle", "table"]:
        return 10  # high-cost
    else:
        return 1  # free space
```



## Visual SLAM Using Computer Vision

Now its time to use camera images instead of range sensors to perform SLAM — Simultaneous Localization and Mapping. Unlike traditional SLAM, which relies on sensors like LiDAR or ToF, Visual SLAM (vSLAM) uses the camera to:

- **Estimate** the robot’s position

- **Build** a map of the environment

- **Track** movement over time using images


### Why Replace ToF Sensors?

As I'm sure you've seen, the **ToF sensor** in the basic package is **quite limited** in it's ability to map the environment due to only having a single degree field of vision. Normally this task falls to LiDAR sensors, which can be implemented in the same way albeit with a much greater field of vision.

| Sensor Type | Pros | Cons         |
|------------------|----------------|----------------------------|
| ToF / LiDAR             | Very accurate, direct depth measurement | Expensive, power-hungry, limited range               |
| Camera (vSLAM)            | Cheap, lightweight, widely available  | Requires ML models, less accurate under poor lighting  |

**Visual SLAM** is a cost-effective solution for robotics where weight, power, or budget is a concern.


### Question 3: Which of the following objects should be treated as an obstacle (infinite cost) in most indoor autonomous navigation?
​ 
**A.** Floor  
**B.** Wall  
**C.** Carpet  

In [ ]:
answer_3 = ""

check_answer("Q3", answer_3)

### How Visual SLAM Works

1. **Feature Detection**
    - The algorithm detects key points (like corners) in camera frames.

2. **Tracking Over Time**
    - As the robot moves, it tracks how those key points shift between frames.

3. **Pose Estimation**
    - Using camera motion and depth estimation, it estimates how far the robot has moved.

4. **Map Building**
    - The camera feed and estimated positions are used to build a sparse or dense map.



## Motion Planning: How Does the Rover Decide Where to Go?

Once localization and mapping are complete, the rover must determine the **best path** to its goal, this is referred to as **Trajectory**. The white line below symbolizes the trajectory of a cleaning rover.

![Trajectory](Trajectory.png)

### Why is Motion Planning Important?

Motion planning allows the rover to **move efficiently** while **avoiding obstacles detected previously**. The rover must decide when to turn, stop, or re-route based on real-time data. Using the data collected we can plan a path that prevents unnecessary detours or collisions.

### How Does Motion Planning Work?

1. **Graph-Based Pathfinding**:

- The map is converted into a graph with nodes.
- The rover selects the lowest-cost path using algorithms like A* or Dijkstra's Algorithm.

2. **Trajectory Optimization**:

- The rover follows a smooth, efficient path rather than just a straight-line connection between points.
- Uses feedback control to adjust for errors in real time.

3. **Reactive Navigation**:

- The rover continuously checks for new obstacles while moving.
- If a new obstacle appears, it recalculates the path.

For example, if an obstacle **blocks** the planned route, the rover **detects** it using ToF sensors, **updates** its map, and **recalculates** the shortest path to the goal.

## Putting It All Together: (v)SLAM + Path Planning

1. The rover starts with an unknown environment.
2. It builds a map while localizing itself using SLAM.
3. Using A*, Dijkstra, or another algorithm, it plans the shortest, safest path to the goal.
4. The rover navigates, continuously updating its position and map.

![Motion Planning](MotionPlanning.png)

## Objective

Use a monocular camera to navigate a real environment, avoid obstacles, and reach a destination.



### What Your Robot Must Do

1. See objects in the environment using your trained YOLO model

2. Estimate depth to understand where those objects are located

3. Map the space and track its position using visual SLAM

4. Assign heuristic values to detected objects (walls, chairs, people, etc.)

5. Plan a path through free space using the A* algorithm

6. Navigate in real time, reacting to new detections along the way



### Scenario Example

You set your rover down in a cluttered indoor space with a visible goal marker and it must:

1. Identify obstacles (like chairs and people)

2. Avoid them based on their heuristic cost

3. Estimate the path to the goal

4. Update the map as it goes

5. Recalculate the route if conditions change



### Requirements

- Camera must be used for both object detection and SLAM

- Robot must build and update a visual map

- A path must be planned and followed, not just reactive movement

- Logs or visuals of:

- Detected objects

- Pathfinding heatmap or tree

- Final trajectory and map

# Conclusion

In this final module, we brought together everything you’ve learned to build a complete autonomous driving system using computer vision. You now have the skills to:

- Detect and classify objects using a YOLO model
- Estimate distances with monocular depth estimation
- Create a visual map of the environment using Visual SLAM
- Assign heuristic values to objects for pathfinding
- Plan and follow efficient paths using the A* algorithm

These techniques simulate the logic behind real-world self-driving systems and give your rover the ability to “see,” think, and move autonomously. From detecting a chair to rerouting around it in real time, your robot now makes smart decisions based on visual input. This is a huge leap from where you started — and a major step toward mastering autonomous robotics.








